In [ ]:
import sys
sys.path.append("../")
import numpy as np
import matplotlib.pyplot as plt

from NMSSE.lowRank.NMLRSSE_Strang_CUDA import NMLRSSE_Strang_CUDA
from lowRank import compute_expectation as ce
from utils.noise_generator import ColoredNoiseGenerator_Cholesky
from utils import multi_index

# bath correlation function in Cai_2020_CPAM
Delta = 1
beta = 5/Delta
wc = 2.5 * Delta
wmax = 4 * wc
CapL = 200
factor = 1 - np.exp(-wmax/wc)
wl = -wc * np.log(1 - np.linspace(1, CapL, CapL)/CapL * factor)
cl = wl * np.sqrt((0.2 * wc/CapL) * factor)

_coth = 1.0 / np.tanh(0.5 * beta * wl)
_pref = (cl**2) / (2.0 * wl)
def bath_corr(t):
    t_arr = np.asarray(t)
    t1 = np.atleast_1d(t_arr).astype(float)
    phase = wl[:, None] * t1[None, :]             # shape: (CapL, len(t))
    out = np.sum(_pref[:, None] * (np.cos(phase) * _coth[:, None] - 1j * np.sin(phase)), axis=0)
    return out[0] if t_arr.ndim == 0 else out

# spin-boson model parameters
eps = [0, Delta, 2*Delta]
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
Hs = [Delta*sx + e*sz for e in eps]
L = sz

In [ ]:
# parameters for the simulation
tmax = 5.0 / Delta
N_steps = 100
psi0 = np.array([1.0, 0.0], dtype=complex)
N_traj = 10000
rank = 10
# hierarchy truncation (total order cutoff for the multi-index set)
# set smaller than N_steps to avoid combinatorial blow-up when rank grows
max_layer = 8
parallel_traj = True

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for idx in range(3):
    NMLRSSE_solver = NMLRSSE_Strang_CUDA(
        Hs = Hs[idx],
        L = L,
        bath_corr = bath_corr,
        tmax = tmax,
        N_steps = N_steps,
        rank = rank,
        max_layer = max_layer,
    )
    psis = NMLRSSE_solver.solve(N_traj, psi0)
    sigma_z = ce.compute_expectation_value(psis, sz)
    axes[idx].plot(NMLRSSE_solver.t_grid, sigma_z, label='rank = {}'.format(rank))
    data = np.load("ref_spinboson/eps_{}.npz".format(eps[idx]))
    tgrid = data['t']
    sigmaz = data['sz']
    axes[idx].plot(tgrid, sigmaz, label='QuAPI')
    axes[idx].set_xlabel('t')
    axes[idx].set_ylabel(r'$\langle \sigma_z \rangle$')
    axes[idx].set_title(r'$\epsilon$={}'.format(eps[idx]))
    axes[idx].grid()
    axes[idx].legend()
    test_op = sigma_z[::(N_steps//50)]
    error = np.linalg.norm(test_op - sigmaz) / np.linalg.norm(sigmaz)
    print("Relative error for eps={}: {:.3e}".format(eps[idx], error))

plt.suptitle('N_traj={}, max_layer={}, N_steps={}'.format(N_traj, max_layer, N_steps))
plt.tight_layout()
plt.show()

In [ ]:
import importlib
import time
import numpy as np

# Reload module to ensure the latest solver implementation is used.
import lowRank.NMLRSSE_Strang_CUDA as strang_cuda_mod
importlib.reload(strang_cuda_mod)
NMLRSSE_Strang_CUDA = strang_cuda_mod.NMLRSSE_Strang_CUDA

# Final benchmark configuration (best measured on this A10 setup).
BEST_THREADS = (128, 4)
BEST_BATCH = 64
N_REPEAT = 3

solver = NMLRSSE_Strang_CUDA(
    Hs=Hs[0],
    L=L,
    bath_corr=bath_corr,
    tmax=tmax,
    N_steps=N_steps,
    rank=rank,
    max_layer=max_layer,
)

# Warm-up to exclude first-call JIT overhead from the benchmark.
_ = solver.solve(
    N_traj=min(8, N_traj),
    psi0=psi0,
    backend="numba-gpu",
    threadsperblock=BEST_THREADS,
    traj_batch_size=BEST_BATCH,
)

timings = []
for _ in range(N_REPEAT):
    t0 = time.perf_counter()
    _ = solver.solve(
        N_traj=N_traj,
        psi0=psi0,
        backend="numba-gpu",
        threadsperblock=BEST_THREADS,
        traj_batch_size=BEST_BATCH,
    )
    timings.append(time.perf_counter() - t0)

avg_time = float(np.mean(timings))
print(f"GPU benchmark avg time over {N_REPEAT} runs: {avg_time:.4f} s")